# ForestGuard — 03 Análisis exploratorio
Estadísticas descriptivas, distribuciones, visualizaciones y relaciones entre variables.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
archivo_processed = Path("data/processed/incendios_forestales_2025_limpio.csv")
if not archivo_processed.exists():
    raise FileNotFoundError("Ejecuta primero 02_limpieza_transformacion.ipynb.")
df_limpio = pd.read_csv(archivo_processed)
df = df_limpio.copy()
print("Dataset procesado cargado:", df.shape)


## 3. Estadísticas descriptivas

In [ ]:
variables_numericas = [
    "latitud", "longitud", "mes", "hadecobert",
    "personasaf", "familiasaf", "animalesaf", "animalesmu"
]

variables_numericas = [c for c in variables_numericas if c in df.columns]
df[variables_numericas].describe().T


### Interpretación

La variable `hadecobert`, que representa las hectáreas de cobertura vegetal afectada, tiene una media aproximada de **2,90 hectáreas**, una mediana de **0,15 hectáreas** y un valor máximo de **105 hectáreas**. Esto muestra que la mayoría de los incendios tuvieron una afectación pequeña, aunque existen algunos casos con valores mucho mayores.


## 4. Valores faltantes

In [ ]:
faltantes_iniciales = pd.DataFrame({
    "Valores faltantes": df.isnull().sum(),
    "Porcentaje": (df.isnull().mean() * 100).round(2)
})

faltantes_iniciales[faltantes_iniciales["Valores faltantes"] > 0]


En esta primera revisión no aparecen valores `NaN`. Sin embargo, algunos campos de texto pueden estar vacíos y guardados como cadenas `""`, por lo que también es necesario revisar esos casos.


### 4.1 Revisión de campos vacíos en variables de texto

In [ ]:
vacios = {}

for columna in df.select_dtypes(include="object").columns:
    cantidad = (
        df[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    if cantidad > 0:
        vacios[columna] = cantidad

campos_vacios = pd.Series(vacios).sort_values(ascending=False)
campos_vacios


### Interpretación

La revisión muestra que sí existen campos vacíos en varias variables de texto. Estos valores no eran reconocidos inicialmente como `NaN`, por lo que durante la limpieza se convertirán a datos faltantes reales para poder identificarlos correctamente.


## 5. Registros duplicados

In [ ]:
duplicados = df.duplicated().sum()
print("Cantidad de registros duplicados:", duplicados)


### Interpretación

No se encontraron registros completamente duplicados en el dataset, por lo que no es necesario eliminar filas por este motivo.


## 6. Revisión de variables categóricas

In [ ]:
columnas_categoricas = [
    "provincia", "canton", "parroquia", "causa",
    "niveldelev", "nivelinf", "region", "evento"
]

for columna in columnas_categoricas:
    if columna in df.columns:
        print(f"\n--- {columna.upper()} ---")
        print(df[columna].value_counts(dropna=False).head(20))


### Interpretación

La variable `causa` presenta una limitación importante, ya que **426 de los 435 registros** aparecen como “Desconocida”. Por esta razón, no se considera una variable fuerte para una futura etapa de Machine Learning.

También se observa que `nivelinf` contiene tres categorías: **Nivel 1, Nivel 2 y Nivel 3**. Esta variable podría servir como objetivo de clasificación, aunque existe un desbalance entre sus categorías.


## 7. Visualizaciones exploratorias

### 7.1 Incendios por provincia

In [ ]:
conteo_provincia = df["provincia"].value_counts().head(15).sort_values()

plt.figure(figsize=(9, 6))
conteo_provincia.plot(kind="barh")
plt.title("Incendios forestales registrados por provincia")
plt.xlabel("Número de registros")
plt.ylabel("Provincia")
plt.tight_layout()
plt.show()


### Interpretación

Dentro del dataset analizado, Guayas y Pichincha concentran la mayor cantidad de registros. Esto muestra que los incendios registrados no están distribuidos de forma uniforme entre las provincias.


### 7.2 Incendios por mes

In [ ]:
conteo_mes = df["mes"].value_counts().sort_index()

plt.figure(figsize=(9, 5))
conteo_mes.plot(kind="bar")
plt.title("Incendios forestales registrados por mes")
plt.xlabel("Mes")
plt.ylabel("Número de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Interpretación

La gráfica permite ver cómo se distribuyen los registros entre los meses disponibles en el dataset y si existen periodos con mayor concentración de incendios.


### 7.3 Principales causas registradas

In [ ]:
causas = (
    df["causa"]
    .fillna("Sin información")
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(9, 6))
causas.plot(kind="barh")
plt.title("Principales causas registradas")
plt.xlabel("Número de registros")
plt.ylabel("Causa")
plt.tight_layout()
plt.show()


### Interpretación

La mayoría de los registros aparecen con la causa “Desconocida”. Esto confirma que la variable tiene poca variedad y puede aportar poca información a un futuro modelo.


### 7.4 Distribución de hectáreas afectadas

In [ ]:
area = pd.to_numeric(df["hadecobert"], errors="coerce").dropna()

plt.figure(figsize=(9, 5))
plt.hist(area, bins=30)
plt.title("Distribución de hectáreas de cobertura vegetal afectada")
plt.xlabel("Hectáreas")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


### Interpretación

La mayoría de los incendios presentan pocas hectáreas afectadas, mientras que algunos registros tienen valores mucho mayores. Estos casos se revisarán como posibles valores atípicos, pero no se eliminarán automáticamente.


### 7.5 Distribución de nivel de afectación

In [ ]:
nivel = df["nivelinf"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
nivel.plot(kind="bar")
plt.title("Distribución del nivel de afectación")
plt.xlabel("Nivel")
plt.ylabel("Número de registros")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Interpretación

`nivelinf` presenta un desbalance claro entre sus categorías:

- Nivel 1: **356 registros**
- Nivel 2: **70 registros**
- Nivel 3: **9 registros**

Esto deberá tomarse en cuenta si en una siguiente etapa se desarrolla un modelo de clasificación.


## 8. Matriz de correlación

In [ ]:
variables_correlacion = [
    "latitud",
    "longitud",
    "mes",
    "hadecobert",
    "feriado"
]

variables_correlacion = [c for c in variables_correlacion if c in df.columns]

correlacion = df[variables_correlacion].corr()
correlacion


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(correlacion, aspect="auto")
plt.xticks(range(len(correlacion.columns)), correlacion.columns, rotation=90)
plt.yticks(range(len(correlacion.columns)), correlacion.columns)
plt.colorbar()
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()


### Interpretación

La matriz ayuda a revisar si existe alguna relación lineal entre las variables numéricas seleccionadas. No se utiliza como prueba de causalidad, sino como una referencia inicial para conocer mejor los datos.
